# ICICLE vs. NEIMS vs. RASSP vs. MassFormer retrieval & similarity — structural analysis

At the global (unfiltered) PubChem retrieval level, NEIMS outperforms ICICLE
substantially (autofail top-1: NEIMS 0.158 vs. ICICLE 0.120), despite the two
models having near-identical (ICICLE actually somewhat *better* on 5 of 7
metrics) mean predicted-spectrum similarity to ground truth. This notebook
investigates whether retrieval and similarity gaps between models are
structured — do they depend on molecular weight, ring systems, heteroatom
content, rotatable bonds, or functional-group content — across **all four
models**: ICICLE, NEIMS, RASSP, MassFormer.

Model colors throughout match `icicle.utils.visualization.eval_plots.model_color`
— the same convention used in `fig_similarity_results.ipynb` (ICICLE=palette[0],
NEIMS=palette[3], RASSP=palette[5], MassFormer=palette[8]).

**Four parts**:

1. **Global PubChem retrieval** (~93.6M candidates, no formula restriction) —
   **ICICLE, NEIMS, MassFormer only** (RASSP has no PubChem-scale
   batch-inference prediction HDF5, so no global-retrieval data exists for it).
   Single seed per model (the seed used to build each model's PubChem
   prediction HDF5).
2. **Formula-match retrieval, random split** — all 4 models, 3-seed mean ±
   95% CI (MassFormer random is missing seed s2 — `mean_ci_t` handles n=2
   gracefully with a wider CI via the t-distribution, so MassFormer random
   uses n=2 while the other 3 models use n=3).
3. **Formula-match retrieval, scaffold split** — all 4 models, 3 seeds each
   (MassFormer has all 3 seeds on scaffold).
4. **Similarity metrics** (all 7: cosine, entropy similarity/distance,
   spectral contrast angle, MSE, weighted cosine, composite) as a function
   of structural descriptors — all 4 models, **both random and scaffold
   split**, violin plots per (metric x descriptor), plus exmol
   functional-group breakdowns.

Every binned table/plot reports **n** (query count) per bin, so sample size
behind each comparison is always visible alongside the summary statistic.

**Data sources** — see the `MODEL_REGISTRY` cell below for the full path
mapping. Key checkpoints: ICICLE random =
`entropy_random_s1/checkpoints/best-model-val_loss=0.1183-epoch=48.ckpt`
(the checkpoint used to build `pubchem_predictions_rerun_260710.hdf5`).
An earlier, INCORRECT similarity comparison used
`results/eval/icicle_rdm_no_xeno_sim/` (a small, ~3-months-older,
335-query dev run) — not used anywhere in this notebook.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import rdMolDescriptors
import exmol
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from matplotlib.patches import Patch

from icicle.utils.visualization.eval_plots import mean_ci_t, model_color
from icicle.utils.visualization.style import (
    get_palette,
    make_fig,
    save_fig,
    set_style,
)

set_style("manuscript")
palette = get_palette()

REPO_ROOT = Path("/home/magled/icicle-dev")
RESULTS = REPO_ROOT / "results"
EVAL = RESULTS / "eval"
OUTPUT_DIR = REPO_ROOT / "figures" / "retrieval_structural_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = ["ICICLE", "NEIMS", "RASSP", "MassFormer"]
GLOBAL_RETRIEVAL_MODELS = [
    "ICICLE",
    "NEIMS",
    "MassFormer",
]  # RASSP has no PubChem HDF5
MODEL_COLORS = {
    m: model_color(m, fallback_index=i) for i, m in enumerate(MODELS)
}
print(MODEL_COLORS)

## Model registry — all result paths in one place

`SIM_DIRS[split][model]` and `FORMULA_DIRS[split][model]` are lists of
per-seed directory paths (containing `similarity_results.csv` /
`retrieval_with_formula_results.csv`); missing seeds are simply shorter
lists (`mean_ci_t` handles n < 3 gracefully). `GLOBAL_RETRIEVAL_DIRS` has
no split axis (only random-split PubChem predictions exist).


In [ ]:
SIM_DIRS = {
    "random": {
        "ICICLE": [EVAL / f"final_entropy_random_s{i}_sim" for i in (1, 2, 3)],
        "NEIMS": [EVAL / f"neims_random_s{i}" for i in (1, 2, 3)],
        "RASSP": [EVAL / f"rassp_random_s{i}" for i in (1, 2, 3)],
        "MassFormer": [
            EVAL / "massformer_random_s1",
            EVAL / "massformer_random_s3",
        ],  # s2 missing
    },
    "scaffold": {
        "ICICLE": [
            EVAL / f"final_entropy_scaffold_s{i}_sim" for i in (1, 2, 3)
        ],
        "NEIMS": [EVAL / f"neims_scaffold_s{i}" for i in (1, 2, 3)],
        "RASSP": [EVAL / f"rassp_scaffold_s{i}" for i in (1, 2, 3)],
        "MassFormer": [EVAL / f"massformer_scaffold_s{i}" for i in (1, 2, 3)],
    },
}

FORMULA_DIRS = {
    "random": {
        "ICICLE": [
            EVAL / f"final_entropy_random_s{i}_retr" for i in (1, 2, 3)
        ],
        "NEIMS": [EVAL / f"neims_random_s{i}" for i in (1, 2, 3)],
        "RASSP": [EVAL / f"rassp_random_s{i}" for i in (1, 2, 3)],
        "MassFormer": [
            EVAL / "massformer_random_s1",
            EVAL / "massformer_random_s3",
        ],  # s2 missing
    },
    "scaffold": {
        "ICICLE": [
            EVAL / f"final_entropy_scaffold_s{i}_retr" for i in (1, 2, 3)
        ],
        "NEIMS": [EVAL / f"neims_scaffold_s{i}" for i in (1, 2, 3)],
        "RASSP": [EVAL / f"rassp_scaffold_s{i}" for i in (1, 2, 3)],
        "MassFormer": [EVAL / f"massformer_scaffold_s{i}" for i in (1, 2, 3)],
    },
}

GLOBAL_RETRIEVAL_DIRS = {
    "ICICLE": RESULTS / "pubchem_retrieval_eval_icicle_rerun_260710",
    "NEIMS": RESULTS / "pubchem_retrieval_eval_neims",
    "MassFormer": RESULTS / "pubchem_retrieval_eval_massformer",
}

for split in SIM_DIRS:
    for model, dirs in SIM_DIRS[split].items():
        existing = [d for d in dirs if (d / "similarity_results.csv").exists()]
        print(
            f"[sim/{split}] {model}: {len(existing)}/{len(dirs)} seeds present"
        )

for split in FORMULA_DIRS:
    for model, dirs in FORMULA_DIRS[split].items():
        existing = [
            d
            for d in dirs
            if (d / "retrieval_with_formula_results.csv").exists()
        ]
        print(
            f"[formula/{split}] {model}: {len(existing)}/{len(dirs)} seeds present"
        )

## Shared setup: structural descriptors + functional groups

Computed once per unique test molecule (keyed by `mol_id`), reused across
all four parts below. Restricted up front to only the molecules that
actually appear in the query sets used anywhere in this notebook.


In [ ]:
metadata_full = pd.read_csv(
    REPO_ROOT / "data" / "NIST2023_GCMS_main" / "metadata.tsv",
    sep="\t",
    usecols=["mol_id", "mw", "standardized_smiles", "formula", "inchi_key"],
)
metadata_full["inchikey14"] = metadata_full["inchi_key"].astype(str).str[:14]
print(f"metadata.tsv total rows: {len(metadata_full)}")

_icicle_global_ids = pd.read_csv(
    GLOBAL_RETRIEVAL_DIRS["ICICLE"] / "retrieval_global_per_query.tsv",
    sep="\t",
    usecols=["mol_id"],
)["mol_id"]
_neims_global_ids = pd.read_csv(
    GLOBAL_RETRIEVAL_DIRS["NEIMS"] / "retrieval_global_per_query.tsv",
    sep="\t",
    usecols=["mol_id"],
)["mol_id"]
_massformer_global_ids = pd.read_csv(
    GLOBAL_RETRIEVAL_DIRS["MassFormer"] / "retrieval_global_per_query.tsv",
    sep="\t",
    usecols=["mol_id"],
)["mol_id"]
_icicle_formula_ids = pd.read_csv(
    FORMULA_DIRS["random"]["ICICLE"][0] / "retrieval_with_formula_results.csv",
    usecols=["spec"],
)["spec"].rename("mol_id")

_needed_mol_ids = (
    set(_icicle_global_ids)
    | set(_neims_global_ids)
    | set(_massformer_global_ids)
    | set(_icicle_formula_ids)
)
print(
    f"Molecules actually needed (union of query sets seen so far): {len(_needed_mol_ids)}"
)

metadata = metadata_full[
    metadata_full["mol_id"].isin(_needed_mol_ids)
].reset_index(drop=True)
print(f"Restricted metadata rows: {len(metadata)}")


def compute_descriptors(smiles: str) -> dict:
    """RDKit structural descriptors + exmol functional groups for one SMILES string."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {
            "heavy_atoms": np.nan,
            "num_rings": np.nan,
            "num_aromatic_rings": np.nan,
            "num_rotatable_bonds": np.nan,
            "num_heteroatoms": np.nan,
            "functional_groups": frozenset(),
        }
    num_heteroatoms = sum(
        1 for atom in mol.GetAtoms() if atom.GetAtomicNum() not in (1, 6)
    )
    return {
        "heavy_atoms": mol.GetNumHeavyAtoms(),
        "num_rings": rdMolDescriptors.CalcNumRings(mol),
        "num_aromatic_rings": rdMolDescriptors.CalcNumAromaticRings(mol),
        "num_rotatable_bonds": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "num_heteroatoms": num_heteroatoms,
        "functional_groups": frozenset(
            exmol.get_functional_groups(mol, return_all=True)
        ),
    }


desc_records = []
for mol_id, smiles in tqdm(
    zip(metadata["mol_id"], metadata["standardized_smiles"]),
    total=len(metadata),
    desc="RDKit + exmol descriptors",
):
    d = compute_descriptors(smiles)
    d["mol_id"] = mol_id
    desc_records.append(d)

descriptors = pd.DataFrame(desc_records)
print(
    f"RDKit parse failures: {descriptors['heavy_atoms'].isna().sum()} / {len(descriptors)}"
)
descriptors.head()


def ensure_descriptors_for(mol_ids: set) -> None:
    """Extend the global `descriptors`/`metadata` tables in place to cover
    any additional mol_ids not already computed (e.g. scaffold-split-only
    molecules not seen in Part 1/2's query sets)."""
    global descriptors, metadata
    missing = mol_ids - set(descriptors["mol_id"])
    if not missing:
        return
    extra_meta = metadata_full[metadata_full["mol_id"].isin(missing)]
    extra_records = []
    for mol_id, smiles in tqdm(
        zip(extra_meta["mol_id"], extra_meta["standardized_smiles"]),
        total=len(extra_meta),
        desc="RDKit + exmol descriptors (extra molecules)",
    ):
        d = compute_descriptors(smiles)
        d["mol_id"] = mol_id
        extra_records.append(d)
    descriptors = pd.concat(
        [descriptors, pd.DataFrame(extra_records)], ignore_index=True
    )
    metadata = pd.concat(
        [metadata, extra_meta], ignore_index=True
    ).drop_duplicates("mol_id")
    print(
        f"descriptors now covers {len(descriptors)} molecules (+{len(missing)})"
    )

## Shared helper functions

Generalized to N models (a `{model: value}` dict) rather than hardcoded
ICICLE/NEIMS pairs, reused across all four parts.


In [ ]:
HEAVY_ATOM_BINS = [0, 6, 8, 10, 12, 14, 18, 100]
RING_BINS = [-0.5, 0.5, 1.5, 2.5, 10]
AROMATIC_RING_BINS = [-0.5, 0.5, 1.5, 10]
ROTATABLE_BINS = [-0.5, 0.5, 2.5, 4.5, 100]
HETEROATOM_BINS = [-0.5, 0.5, 1.5, 2.5, 20]
MW_BINS = [0, 100, 150, 200, 250, 300, 400, 1000]

DESCRIPTOR_SPECS = [
    ("mw", MW_BINS, "Molecular weight (Da)"),
    ("heavy_atoms", HEAVY_ATOM_BINS, "Heavy atoms"),
    ("num_rings", RING_BINS, "Ring count"),
    ("num_rotatable_bonds", ROTATABLE_BINS, "Rotatable bonds"),
    ("num_heteroatoms", HETEROATOM_BINS, "Heteroatom count"),
]
DESCRIPTOR_COLS = [
    "mw",
    "heavy_atoms",
    "num_rings",
    "num_aromatic_rings",
    "num_rotatable_bonds",
    "num_heteroatoms",
]
SIMILARITY_METRICS = [
    "cosine_similarity",
    "entropy_similarity",
    "entropy_distance",
    "spectral_contrast_angle",
    "mean_squared_error",
    "weighted_cosine_nist_gc",
    "composite_similarity_nist_gc",
]


def top1_by_bins_n(df, col, bins, top1_cols: dict):
    """Top-1 rate per model (single seed each), binned, WITH per-bin n.

    top1_cols: {model: column_name_of_bool_top1}.
    Returns one row per bin: n, then <model>_top1_rate for each model.
    """
    binned = pd.cut(df[col], bins=bins)
    agg = {"n": ("mol_id", "count")}
    for model, colname in top1_cols.items():
        agg[f"{model}_top1_rate"] = (colname, "mean")
    return df.groupby(binned, observed=True).agg(**agg).round(4)


def plot_topk_panels_single_seed(
    df, specs, top1_cols: dict, output_stem_prefix, output_dir
):
    """One figure per descriptor: top-1 rate (%) per model, single seed
    (no CI — used for Part 1's global retrieval, which has exactly one
    seed per model)."""
    for col, bins, xlabel in specs:
        summary = top1_by_bins_n(df, col, bins, top1_cols)
        fig, ax = make_fig("square")
        x = np.arange(len(summary))
        for model in top1_cols:
            ax.plot(
                x,
                summary[f"{model}_top1_rate"] * 100,
                marker="o",
                color=MODEL_COLORS[model],
                label=model,
            )
        ax.set_xticks(x)
        ax.set_xticklabels(
            [
                f"{b}\n(n={n})"
                for b, n in zip(summary.index.astype(str), summary["n"])
            ],
            rotation=45,
            ha="right",
            fontsize=6,
        )
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Top-1 accuracy (%)")
        ax.legend(frameon=False)
        stem = f"{output_stem_prefix}_{col}"
        save_fig(fig, stem, output_dir)
        print(f"Saved {stem}.[svg|png]")


def wide_per_seed(seed_frames: dict, id_col: str, prefix: str) -> tuple:
    """Merge {seed_name: df} into one wide df with one top1 column per
    seed, keeping only rows present in ALL given seeds."""
    wide = None
    for seed, df in seed_frames.items():
        df = df.rename(columns={"top1": f"{prefix}_top1_{seed}"})
        wide = df if wide is None else wide.merge(df, on=id_col, how="outer")
    seed_cols = [f"{prefix}_top1_{seed}" for seed in seed_frames]
    n_seeds_present = wide[seed_cols].notna().sum(axis=1)
    wide = wide[n_seeds_present == len(seed_frames)].reset_index(drop=True)
    return wide, seed_cols


def load_icicle_formula_top1(path: Path) -> pd.DataFrame:
    """One row per query: mol_id, top1 (bool) for this seed."""
    raw = pd.read_csv(path / "retrieval_with_formula_results.csv")
    true_rows = raw[~raw["is_decoy"]].rename(columns={"spec": "mol_id"})
    true_rows["top1"] = true_rows["rank_cosine_similarity"] == 1
    return true_rows[["mol_id", "top1"]]


def load_baseline_formula_top1(path: Path) -> pd.DataFrame:
    """One row per query: inchikey14, top1 (bool) for this seed.

    Shared loader for NEIMS/RASSP/MassFormer — same query_inchikey14 /
    is_correct schema. A query counts as top-1 if ANY of its is_correct
    rows achieved rank 1 (duplicate InChIKey14 candidates can occur).
    """
    raw = pd.read_csv(path / "retrieval_with_formula_results.csv")
    true_rows = raw[raw["is_correct"]].rename(
        columns={"query_inchikey14": "inchikey14"}
    )
    true_rows["top1"] = true_rows["rank_cosine_similarity"] == 1
    return true_rows.groupby("inchikey14", as_index=False)["top1"].max()


def build_formula_wide(dirs_by_model: dict, prefix_map: dict) -> dict:
    """For each model, load its per-seed top1 frames and merge into a
    wide (one column per seed) frame requiring all seeds present.

    Returns {model: (wide_df, seed_cols, id_col)}.
    """
    out = {}
    for model, dirs in dirs_by_model.items():
        seed_frames = {}
        for d in dirs:
            seed = (
                d.name.split("_")[-1]
                if d.name[-1].isdigit() or d.name[-2:].startswith("s")
                else d.name
            )
            loader = (
                load_icicle_formula_top1
                if model == "ICICLE"
                else load_baseline_formula_top1
            )
            seed_frames[d.name] = loader(d)
        id_col = "mol_id" if model == "ICICLE" else "inchikey14"
        wide, seed_cols = wide_per_seed(seed_frames, id_col, prefix_map[model])
        out[model] = (wide, seed_cols, id_col)
        print(
            f"  {model}: {len(wide)} queries present in all {len(dirs)} seeds"
        )
    return out


def top1_by_bins_ci_n(df, col, bins, seed_cols_by_model: dict):
    """Per-bin top-1 rate (mean + 95% CI half-width across seeds) per
    model, plus n (query count in that bin). seed_cols_by_model:
    {model: [seed_col, ...]}.
    """
    binned = pd.cut(df[col], bins=bins)
    rows = []
    for bin_label, group in df.groupby(binned, observed=True):
        row = {"bin": bin_label, "n": len(group)}
        for model, seed_cols in seed_cols_by_model.items():
            seed_rates = [group[c].mean() for c in seed_cols if c in group]
            mean, ci = (
                mean_ci_t(np.array(seed_rates))
                if seed_rates
                else (np.nan, np.nan)
            )
            row[f"{model}_mean"] = mean
            row[f"{model}_ci"] = ci
        rows.append(row)
    return pd.DataFrame(rows).set_index("bin").round(4)


def plot_descriptor_panels_ci_n(
    df, specs, seed_cols_by_model: dict, output_stem_prefix, output_dir
):
    for col, bins, xlabel in specs:
        summary = top1_by_bins_ci_n(df, col, bins, seed_cols_by_model)
        fig, ax = make_fig("square")
        x = np.arange(len(summary))
        for model in seed_cols_by_model:
            ax.errorbar(
                x,
                summary[f"{model}_mean"] * 100,
                yerr=summary[f"{model}_ci"] * 100,
                marker="o",
                capsize=3,
                color=MODEL_COLORS[model],
                label=model,
            )
        ax.set_xticks(x)
        ax.set_xticklabels(
            [
                f"{b}\n(n={n})"
                for b, n in zip(summary.index.astype(str), summary["n"])
            ],
            rotation=45,
            ha="right",
            fontsize=6,
        )
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Top-1 accuracy (%, mean \u00b1 95% CI)")
        ax.legend(frameon=False)
        stem = f"{output_stem_prefix}_{col}"
        save_fig(fig, stem, output_dir)
        print(f"Saved {stem}.[svg|png]")


def functional_group_summary_n(df, rate_cols: dict, min_count=30):
    """Explode functional_groups, aggregate mean top1-rate per group per
    model, plus n. rate_cols: {model: column_name}."""
    cols = ["mol_id", "functional_groups"] + list(rate_cols.values())
    exploded = (
        df[cols]
        .explode("functional_groups")
        .dropna(subset=["functional_groups"])
    )
    agg = {"n": ("mol_id", "count")}
    for model, colname in rate_cols.items():
        agg[f"{model}_rate"] = (colname, "mean")
    summary = exploded.groupby("functional_groups").agg(**agg)
    summary = summary[summary["n"] >= min_count].round(4)
    return summary


def correlation_table_n(df, descriptor_cols, rate_cols: dict):
    """Spearman correlation of each descriptor against each model's rate
    column, plus each descriptor's own correlation with MW (confounding
    check)."""
    valid = df.dropna(subset=descriptor_cols + list(rate_cols.values()))
    rows = []
    for col in descriptor_cols:
        row = {
            "descriptor": col,
            "spearman_rho_vs_mw": (
                1.0 if col == "mw" else spearmanr(valid[col], valid["mw"])[0]
            ),
        }
        for model, colname in rate_cols.items():
            rho, p = spearmanr(valid[col], valid[colname])
            row[f"{model}_rho"] = rho
            row[f"{model}_p"] = p
        rows.append(row)
    return pd.DataFrame(rows).round(4)

---
# Part 1 — Global PubChem retrieval (unfiltered, ~93.6M candidates)

**ICICLE, NEIMS, MassFormer only** — RASSP has no PubChem-scale
batch-inference prediction HDF5, so no global-retrieval comparison is
possible for it. Single seed per model.


In [ ]:
global_per_query = {
    model: pd.read_csv(
        GLOBAL_RETRIEVAL_DIRS[model] / "retrieval_global_per_query.tsv",
        sep="\t",
    )
    for model in GLOBAL_RETRIEVAL_MODELS
}
for model, df in global_per_query.items():
    print(f"{model}: {len(df)} per-query rows")

common_ids = set.intersection(
    *(set(df["mol_id"]) for df in global_per_query.values())
)
print(
    f"Common mol_ids across all {len(GLOBAL_RETRIEVAL_MODELS)} models: {len(common_ids)}"
)

global_merged = (
    metadata[metadata["mol_id"].isin(common_ids)][["mol_id"]]
    .merge(metadata[["mol_id", "mw", "formula"]], on="mol_id")
    .merge(descriptors, on="mol_id", how="left")
)

top1_cols = {}
for model, df in global_per_query.items():
    sub = df[df["mol_id"].isin(common_ids)][["mol_id", "rank_autofail_cosine"]]
    colname = f"{model}_top1"
    sub[colname] = sub["rank_autofail_cosine"] == 1
    global_merged = global_merged.merge(
        sub[["mol_id", colname]], on="mol_id", how="left"
    )
    top1_cols[model] = colname

print(f"Merged rows: {len(global_merged)}")
global_merged.to_csv(
    OUTPUT_DIR / "merged_top1_structural_global.csv", index=False
)
global_merged.head()

## Global retrieval — MW and structural descriptor binning (top-1 accuracy, with n per bin)

In [ ]:
for col, bins, _ in DESCRIPTOR_SPECS:
    print(f"=== {col} ===")
    display(top1_by_bins_n(global_merged, col, bins, top1_cols))
    print()

In [ ]:
plot_topk_panels_single_seed(
    global_merged, DESCRIPTOR_SPECS, top1_cols, "global_top1_vs", OUTPUT_DIR
)

## Global retrieval — functional group breakdown (exmol)

In [ ]:
global_fg_summary = functional_group_summary_n(global_merged, top1_cols)
global_fg_summary.sort_values(
    f"{GLOBAL_RETRIEVAL_MODELS[1]}_rate", ascending=False
).head(15)

## Global retrieval — functional group breakdown, sorted by n (most-represented groups first)

In [ ]:
global_fg_summary_by_n = global_fg_summary.sort_values("n", ascending=False)
print(f"Total functional groups (n>=30): {len(global_fg_summary_by_n)}")
print("Most-represented 10:")
display(global_fg_summary_by_n.head(10))
print("Least-represented 10 (still n>=30):")
display(global_fg_summary_by_n.tail(10))

## Global retrieval — top-1 accuracy vs. functional-group frequency (full range, binned)

Truncating to the 15 most-represented groups only shows the regime
where NEIMS leads. Binning across the full frequency range (all 177
groups with n>=30) checks whether ICICLE's advantage on rare groups
(seen in the gap table sorted by delta) is a real frequency effect or a
handful of cherry-picked examples.

In [ ]:
n_bins = 8
fg_binned = global_fg_summary.copy()
fg_binned["n_bin"] = pd.qcut(fg_binned["n"], n_bins, duplicates="drop")

bin_summary = (
    fg_binned.groupby("n_bin", observed=True)
    .agg(
        n_groups=("n", "count"),
        median_n=("n", "median"),
        **{
            f"{m}_rate": (f"{m}_rate", "mean") for m in GLOBAL_RETRIEVAL_MODELS
        },
    )
    .sort_values("median_n")
)
display(bin_summary)

fig, ax = make_fig("wide")
x = np.arange(len(bin_summary))
for model in GLOBAL_RETRIEVAL_MODELS:
    ax.plot(
        x,
        bin_summary[f"{model}_rate"] * 100,
        marker="o",
        label=model,
        color=MODEL_COLORS[model],
    )
ax.set_xticks(x)
ax.set_xticklabels(
    [
        f"{int(m):,}\n({int(ng)} groups)"
        for m, ng in zip(bin_summary["median_n"], bin_summary["n_groups"])
    ],
    rotation=0,
)
ax.set_xlabel(
    "Functional-group query count n (median per bin, rare -> common)"
)
ax.set_ylabel("Top-1 accuracy (%, mean across groups in bin)")
ax.legend(frameon=False)
save_fig(fig, "global_fg_top1_by_n_binned", OUTPUT_DIR)
print("Saved global_fg_top1_by_n_binned.[svg|png]")

## Global retrieval — NEIMS-ICICLE delta vs. functional-group frequency (labeled scatter, log x)

One point per functional group (no aggregation/binning), so every axis
value is directly countable. Extremes on each side are labeled by name.

In [ ]:
fig, ax = make_fig("wide")
delta = (
    global_fg_summary["NEIMS_rate"] - global_fg_summary["ICICLE_rate"]
) * 100
ax.scatter(global_fg_summary["n"], delta, color=palette[0], alpha=0.6, s=18)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_xscale("log")
ax.set_xlabel("Functional-group query count n (log scale)")
ax.set_ylabel("NEIMS - ICICLE top-1 accuracy (pp)")

# annotate a few extremes: most ICICLE-favoring (rare) and most NEIMS-favoring (common)
sorted_by_delta = global_fg_summary.assign(delta=delta).sort_values("delta")
icicle_favoring = sorted_by_delta.head(3)
neims_favoring = sorted_by_delta.tail(3)
for name, row in pd.concat([icicle_favoring, neims_favoring]).iterrows():
    ax.annotate(
        name,
        (row["n"], row["delta"]),
        fontsize=6,
        xytext=(4, 4),
        textcoords="offset points",
    )

save_fig(fig, "global_fg_delta_vs_n_scatter_labeled", OUTPUT_DIR)
print("Saved global_fg_delta_vs_n_scatter_labeled.[svg|png]")

from scipy.stats import spearmanr

rho, p = spearmanr(global_fg_summary["n"], delta)
print(f"Spearman rho(n, NEIMS-ICICLE delta) = {rho:.3f}, p = {p:.2e}")

## Global retrieval — rare vs. common functional groups (quartile split)

Simplest possible version of the same story: bottom quartile by n
(rarest ~45 groups) vs. top quartile by n (most common ~45 groups),
mean top-1 accuracy per model in each.

In [ ]:
q25, q75 = global_fg_summary["n"].quantile([0.25, 0.75])
rare = global_fg_summary[global_fg_summary["n"] <= q25]
common = global_fg_summary[global_fg_summary["n"] >= q75]

rare_common_table = pd.DataFrame(
    {
        "n_groups": [len(rare), len(common)],
        "median_n": [rare["n"].median(), common["n"].median()],
        **{
            f"{m}_top1_mean": [
                rare[f"{m}_rate"].mean(),
                common[f"{m}_rate"].mean(),
            ]
            for m in GLOBAL_RETRIEVAL_MODELS
        },
    },
    index=[f"rare (n<={q25:.0f})", f"common (n>={q75:.0f})"],
)
display(rare_common_table.round(4))

fig, ax = make_fig("default")
x = np.arange(2)
width = 0.8 / len(GLOBAL_RETRIEVAL_MODELS)
for i, model in enumerate(GLOBAL_RETRIEVAL_MODELS):
    ax.bar(
        x + i * width,
        rare_common_table[f"{model}_top1_mean"] * 100,
        width=width,
        label=model,
        color=MODEL_COLORS[model],
    )
ax.set_xticks(x + width * (len(GLOBAL_RETRIEVAL_MODELS) - 1) / 2)
ax.set_xticklabels(
    [
        f"Rare\n(n\u2264{q25:.0f}, {len(rare)} groups)",
        f"Common\n(n\u2265{q75:.0f}, {len(common)} groups)",
    ]
)
ax.set_ylabel("Mean top-1 accuracy (%)")
ax.legend(frameon=False)
save_fig(fig, "global_fg_rare_vs_common_bar", OUTPUT_DIR)
print("Saved global_fg_rare_vs_common_bar.[svg|png]")

## Global retrieval — quartile trend + representative named group per quartile

The two-bucket rare-vs-common view above collapses the middle 50% of
groups. This shows all 4 quartiles (so the crossover point and its
monotonicity are visible), each paired with its own largest-n
(most statistically stable) named functional group, so the trend is
grounded in real, checkable chemistry rather than only an aggregate.

In [ ]:
quartile_labels = ["Q1 (rarest)", "Q2", "Q3", "Q4 (most common)"]
fg_q = global_fg_summary.copy()
fg_q["quartile"] = pd.qcut(fg_q["n"], 4, labels=quartile_labels)

quartile_agg = fg_q.groupby("quartile", observed=True).agg(
    n_groups=("n", "count"),
    median_n=("n", "median"),
    **{f"{m}_rate": (f"{m}_rate", "mean") for m in GLOBAL_RETRIEVAL_MODELS},
)

# representative group per quartile: largest-n group WHERE ICICLE beats NEIMS
# (not just largest-n overall) so the example is consistent with the point being made
rep_per_quartile = {}
icicle_win_counts = {}
for q in quartile_labels:
    sub = fg_q[fg_q["quartile"] == q]
    icicle_wins = sub[sub["ICICLE_rate"] > sub["NEIMS_rate"]].sort_values(
        "n", ascending=False
    )
    icicle_win_counts[q] = (len(icicle_wins), len(sub))
    rep_per_quartile[q] = icicle_wins.iloc[0]

print(
    "ICICLE-winning groups per quartile (out of total groups in that quartile):"
)
for q in quartile_labels:
    wins, total = icicle_win_counts[q]
    print(f"  {q}: {wins}/{total}")
print()
print(
    "Representative group per quartile (largest n among ICICLE-winning groups):"
)
display(
    pd.DataFrame(rep_per_quartile).T[
        ["n"] + [f"{m}_rate" for m in GLOBAL_RETRIEVAL_MODELS]
    ]
)

# build 8 clusters: quartile-agg, rep-group, quartile-agg, rep-group, ...
cluster_labels = []
cluster_data = {m: [] for m in GLOBAL_RETRIEVAL_MODELS}
for q in quartile_labels:
    agg_row = quartile_agg.loc[q]
    cluster_labels.append(f"{q} avg\n({int(agg_row['n_groups'])} groups)")
    for m in GLOBAL_RETRIEVAL_MODELS:
        cluster_data[m].append(agg_row[f"{m}_rate"])
    rep_row = rep_per_quartile[q]
    cluster_labels.append(f"{rep_row.name}\n(n={int(rep_row['n']):,})")
    for m in GLOBAL_RETRIEVAL_MODELS:
        cluster_data[m].append(rep_row[f"{m}_rate"])

fig, ax = make_fig("wide")
x = np.arange(len(cluster_labels))
width = 0.8 / len(GLOBAL_RETRIEVAL_MODELS)
for i, model in enumerate(GLOBAL_RETRIEVAL_MODELS):
    ax.bar(
        x + i * width,
        np.array(cluster_data[model]) * 100,
        width=width,
        label=model,
        color=MODEL_COLORS[model],
    )
ax.set_xticks(x + width * (len(GLOBAL_RETRIEVAL_MODELS) - 1) / 2)
ax.set_xticklabels(cluster_labels, rotation=45, ha="right", fontsize=7)
ax.set_ylabel("Top-1 accuracy (%)")
ax.legend(frameon=False)
save_fig(fig, "global_fg_quartile_plus_examples_bar", OUTPUT_DIR)
print("Saved global_fg_quartile_plus_examples_bar.[svg|png]")

## Global retrieval — full functional-group table (all 177 groups, n>=30), exported for SI reference

In [ ]:
full_fg_export = global_fg_summary.sort_values("n", ascending=False).copy()
full_fg_export["delta_NEIMS_minus_ICICLE"] = (
    full_fg_export["NEIMS_rate"] - full_fg_export["ICICLE_rate"]
)
full_fg_export.to_csv(OUTPUT_DIR / "global_fg_full_table_by_n.csv")
print(
    f"Exported {len(full_fg_export)} groups to global_fg_full_table_by_n.csv"
)
display(full_fg_export.head(10))
display(full_fg_export.tail(10))

## Global retrieval — Spearman correlations (descriptor vs. top-1, per model)

In [ ]:
global_correlation = correlation_table_n(
    global_merged, DESCRIPTOR_COLS, top1_cols
)
global_correlation

---
# Part 2 — Formula-match retrieval, RANDOM split (all 4 models)

All 4 models, mean top-1 ± 95% CI across seeds (MassFormer has 2 seeds
here, missing s2; the other 3 models have 3 seeds).


In [ ]:
print("Loading random-split formula-match top-1 per model...")
random_formula_wide = build_formula_wide(
    FORMULA_DIRS["random"],
    prefix_map={
        "ICICLE": "icicle",
        "NEIMS": "neims",
        "RASSP": "rassp",
        "MassFormer": "massformer",
    },
)

In [ ]:
def merge_formula_wide(wide_by_model: dict) -> tuple:
    """Merge per-model wide frames on inchikey14 (join key common to all
    models via metadata), keeping only queries present for every model
    with formula-match data. Returns (merged_df, seed_cols_by_model).
    """
    keyed = {}
    for model, (wide, seed_cols, id_col) in wide_by_model.items():
        if id_col == "mol_id":
            wide = wide.merge(
                metadata_full[["mol_id", "inchikey14"]],
                on="mol_id",
                how="left",
            )
        keyed[model] = (wide, seed_cols)

    common_keys = set.intersection(
        *(set(w["inchikey14"]) for w, _ in keyed.values())
    )
    print(
        f"Common inchikey14 across all {len(keyed)} models: {len(common_keys)}"
    )

    merged = metadata_full[metadata_full["inchikey14"].isin(common_keys)][
        ["mol_id", "inchikey14"]
    ].drop_duplicates("inchikey14")
    seed_cols_by_model = {}
    for model, (wide, seed_cols) in keyed.items():
        cols = ["inchikey14"] + seed_cols
        merged = merged.merge(
            wide[wide["inchikey14"].isin(common_keys)][cols],
            on="inchikey14",
            how="inner",
        )
        seed_cols_by_model[model] = seed_cols
    merged = merged.merge(
        metadata_full[["mol_id", "mw", "formula"]], on="mol_id", how="left"
    )
    return merged, seed_cols_by_model


random_formula_merged, random_seed_cols_by_model = merge_formula_wide(
    random_formula_wide
)
ensure_descriptors_for(set(random_formula_merged["mol_id"]))
random_formula_merged = random_formula_merged.merge(
    descriptors, on="mol_id", how="left"
)

for model, seed_cols in random_seed_cols_by_model.items():
    for c in seed_cols:
        random_formula_merged[f"{model}_ratecol_{c}"] = random_formula_merged[
            c
        ]
    random_formula_merged[f"{model}_rate"] = random_formula_merged[
        seed_cols
    ].mean(axis=1)

print(f"Merged rows: {len(random_formula_merged)}")
random_formula_merged.to_csv(
    OUTPUT_DIR / "merged_top1_structural_formula_match_random_3seed.csv",
    index=False,
)
random_formula_merged.head()

## Random-split formula-match — MW and structural descriptor binning (top-1 accuracy, mean ± 95% CI, with n per bin)

In [ ]:
for col, bins, _ in DESCRIPTOR_SPECS:
    print(f"=== {col} ===")
    display(
        top1_by_bins_ci_n(
            random_formula_merged, col, bins, random_seed_cols_by_model
        )
    )
    print()

In [ ]:
plot_descriptor_panels_ci_n(
    random_formula_merged,
    DESCRIPTOR_SPECS,
    random_seed_cols_by_model,
    "formula_match_random_top1_vs",
    OUTPUT_DIR,
)

## Random-split formula-match — functional group breakdown (exmol)

In [ ]:
random_rate_cols = {model: f"{model}_rate" for model in MODELS}
random_fg_summary = functional_group_summary_n(
    random_formula_merged, random_rate_cols
)
random_fg_summary.sort_values("NEIMS_rate", ascending=False).head(15)

## Random-split formula-match — Spearman correlations (descriptor vs. rate, per model)

In [ ]:
random_formula_correlation = correlation_table_n(
    random_formula_merged, DESCRIPTOR_COLS, random_rate_cols
)
random_formula_correlation

---
# Part 3 — Formula-match retrieval, SCAFFOLD split (all 4 models, 3 seeds each)

Same analysis as Part 2 but on the harder scaffold-split generalization
test — all 4 models have all 3 seeds here.


In [ ]:
print("Loading scaffold-split formula-match top-1 per model...")
scaffold_formula_wide = build_formula_wide(
    FORMULA_DIRS["scaffold"],
    prefix_map={
        "ICICLE": "icicle",
        "NEIMS": "neims",
        "RASSP": "rassp",
        "MassFormer": "massformer",
    },
)

In [ ]:
scaffold_formula_merged, scaffold_seed_cols_by_model = merge_formula_wide(
    scaffold_formula_wide
)
ensure_descriptors_for(set(scaffold_formula_merged["mol_id"]))
scaffold_formula_merged = scaffold_formula_merged.merge(
    descriptors, on="mol_id", how="left"
)

for model, seed_cols in scaffold_seed_cols_by_model.items():
    scaffold_formula_merged[f"{model}_rate"] = scaffold_formula_merged[
        seed_cols
    ].mean(axis=1)

print(f"Merged rows: {len(scaffold_formula_merged)}")
scaffold_formula_merged.to_csv(
    OUTPUT_DIR / "merged_top1_structural_formula_match_scaffold_3seed.csv",
    index=False,
)
scaffold_formula_merged.head()

## Scaffold-split formula-match — MW and structural descriptor binning (top-1 accuracy, mean ± 95% CI, with n per bin)

In [ ]:
for col, bins, _ in DESCRIPTOR_SPECS:
    print(f"=== {col} ===")
    display(
        top1_by_bins_ci_n(
            scaffold_formula_merged, col, bins, scaffold_seed_cols_by_model
        )
    )
    print()

In [ ]:
plot_descriptor_panels_ci_n(
    scaffold_formula_merged,
    DESCRIPTOR_SPECS,
    scaffold_seed_cols_by_model,
    "formula_match_scaffold_top1_vs",
    OUTPUT_DIR,
)

## Scaffold-split formula-match — functional group breakdown (exmol)

In [ ]:
scaffold_rate_cols = {model: f"{model}_rate" for model in MODELS}
scaffold_fg_summary = functional_group_summary_n(
    scaffold_formula_merged, scaffold_rate_cols
)
scaffold_fg_summary.sort_values("NEIMS_rate", ascending=False).head(15)

## Scaffold-split formula-match — Spearman correlations (descriptor vs. rate, per model)

In [ ]:
scaffold_formula_correlation = correlation_table_n(
    scaffold_formula_merged, DESCRIPTOR_COLS, scaffold_rate_cols
)
scaffold_formula_correlation

## Random split vs. scaffold split vs. global: overall rate summary (n included)


In [ ]:
summary_rows = []
for model in GLOBAL_RETRIEVAL_MODELS:
    summary_rows.append(
        {
            "regime": "Global retrieval",
            "model": model,
            "n": len(global_merged),
            "top1_rate": global_merged[top1_cols[model]].mean(),
        }
    )
for model in MODELS:
    seed_cols = random_seed_cols_by_model[model]
    rates = np.array([random_formula_merged[c].mean() for c in seed_cols])
    mean, ci = mean_ci_t(rates)
    summary_rows.append(
        {
            "regime": "Formula-match (random)",
            "model": model,
            "n": len(random_formula_merged),
            "top1_rate": mean,
            "ci_95": ci,
            "n_seeds": len(seed_cols),
        }
    )
for model in MODELS:
    seed_cols = scaffold_seed_cols_by_model[model]
    rates = np.array([scaffold_formula_merged[c].mean() for c in seed_cols])
    mean, ci = mean_ci_t(rates)
    summary_rows.append(
        {
            "regime": "Formula-match (scaffold)",
            "model": model,
            "n": len(scaffold_formula_merged),
            "top1_rate": mean,
            "ci_95": ci,
            "n_seeds": len(seed_cols),
        }
    )

overall_summary = pd.DataFrame(summary_rows).round(4)
overall_summary

---
# Part 4 — Similarity metrics as a function of structural descriptors

All 4 models, **both random and scaffold split**, all 7 metrics. Violin
plots per (metric x descriptor), plus exmol functional-group breakdowns.
Each split's seed count follows `SIM_DIRS` above (MassFormer random has
only 2 seeds; everything else has 3).

Two of the seven metrics are DISTANCES/ERRORS (lower = better), not
similarities (higher = better): `entropy_distance` and
`mean_squared_error` — read each panel's y-axis independently.


In [ ]:
def load_similarity_all_seeds(dirs: list, mol_ids: set) -> pd.DataFrame:
    """Concatenate per-seed similarity_results.csv for one model, restricted
    to mol_ids, tagging each row with its seed (directory name)."""
    frames = []
    for d in dirs:
        df = pd.read_csv(
            d / "similarity_results.csv",
            usecols=["mol_id"] + SIMILARITY_METRICS,
        )
        df = df[df["mol_id"].isin(mol_ids)].copy()
        df["seed"] = d.name
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def build_similarity_long(split: str) -> pd.DataFrame:
    """One row per (model, mol_id, seed), all 7 metrics as columns —
    'long' format so seed-level violin plots naturally include every
    per-seed observation, not just a pre-averaged mean."""
    common_ids = set.intersection(
        *(
            set(
                pd.read_csv(
                    dirs[0] / "similarity_results.csv", usecols=["mol_id"]
                )["mol_id"]
            )
            for dirs in SIM_DIRS[split].values()
        )
    )
    print(
        f"[{split}] common mol_ids (first seed of each model): {len(common_ids)}"
    )
    frames = []
    for model, dirs in SIM_DIRS[split].items():
        df = load_similarity_all_seeds(dirs, common_ids)
        df["model"] = model
        frames.append(df)
    long_df = pd.concat(frames, ignore_index=True)
    # Restrict to mol_ids present for ALL models (not just first-seed ids).
    ids_per_model = long_df.groupby("model")["mol_id"].apply(set)
    common_all = set.intersection(*ids_per_model)
    long_df = long_df[long_df["mol_id"].isin(common_all)].reset_index(
        drop=True
    )
    print(
        f"[{split}] common mol_ids across all models, all their seeds: {len(common_all)}"
    )
    return long_df.merge(
        metadata_full[["mol_id", "mw", "formula"]], on="mol_id", how="left"
    )


similarity_long = {
    split: build_similarity_long(split) for split in ("random", "scaffold")
}
for split, df in similarity_long.items():
    print(
        f"{split}: {len(df)} rows, {df['mol_id'].nunique()} unique molecules, models={sorted(df['model'].unique())}"
    )

In [ ]:
for split in ("random", "scaffold"):
    ensure_descriptors_for(set(similarity_long[split]["mol_id"]))
    similarity_long[split] = similarity_long[split].merge(
        descriptors, on="mol_id", how="left"
    )
    similarity_long[split].to_csv(
        OUTPUT_DIR / f"merged_similarity_structural_{split}.csv", index=False
    )

In [ ]:
def violin_panels_multi_model(
    df, metric, descriptor_col, bins, models, output_stem, output_dir, xlabel
):
    """One violin-plot figure: x-axis = descriptor bins, each bin shows one
    violin per model (all seed-level observations pooled per model per bin).
    """
    binned = pd.cut(df[descriptor_col], bins=bins)
    bin_cats = binned.cat.categories
    n_bins = len(bin_cats)
    n_models = len(models)
    total_width = 0.8
    width = total_width / n_models

    fig, ax = make_fig("wide")
    bin_ns = []
    for bi, cat in enumerate(bin_cats):
        bin_mask = binned == cat
        bin_ns.append(bin_mask.sum())
        for mi, model in enumerate(models):
            vals = (
                df.loc[bin_mask & (df["model"] == model), metric]
                .dropna()
                .values
            )
            if len(vals) == 0:
                continue
            pos = bi + (mi - (n_models - 1) / 2) * width
            parts = ax.violinplot(
                [vals], positions=[pos], widths=width * 0.9, showmeans=True
            )
            for body in parts["bodies"]:
                body.set_facecolor(MODEL_COLORS[model])
                body.set_alpha(0.6)
                body.set_edgecolor(MODEL_COLORS[model])
            for key in ("cmeans", "cmedians", "cbars", "cmins", "cmaxes"):
                if key in parts:
                    parts[key].set_color(MODEL_COLORS[model])

    ax.set_xticks(np.arange(n_bins))
    ax.set_xticklabels(
        [f"{cat}\n(n={n})" for cat, n in zip(bin_cats, bin_ns)],
        rotation=45,
        ha="right",
        fontsize=6,
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel(metric.replace("_", " "))
    legend_handles = [
        Patch(facecolor=MODEL_COLORS[m], alpha=0.6, label=m) for m in models
    ]
    ax.legend(handles=legend_handles, frameon=False, fontsize=6)

    stem = f"{output_stem}_{metric}_vs_{descriptor_col}"
    save_fig(fig, stem, output_dir)
    print(f"Saved {stem}.[svg|png]")

## Violin plots — RANDOM split (7 metrics x 5 descriptors, all 4 models overlaid per bin)

In [ ]:
for col, bins, xlabel in DESCRIPTOR_SPECS:
    print(f"=== Descriptor: {col} (random split) ===")
    for metric in SIMILARITY_METRICS:
        violin_panels_multi_model(
            similarity_long["random"],
            metric,
            col,
            bins,
            MODELS,
            "similarity_violin_random",
            OUTPUT_DIR,
            xlabel,
        )
    print()

## Violin plots — SCAFFOLD split (7 metrics x 5 descriptors, all 4 models overlaid per bin)

In [ ]:
for col, bins, xlabel in DESCRIPTOR_SPECS:
    print(f"=== Descriptor: {col} (scaffold split) ===")
    for metric in SIMILARITY_METRICS:
        violin_panels_multi_model(
            similarity_long["scaffold"],
            metric,
            col,
            bins,
            MODELS,
            "similarity_violin_scaffold",
            OUTPUT_DIR,
            xlabel,
        )
    print()

## Similarity — functional group breakdown (exmol), both splits

In [ ]:
def functional_group_similarity_summary(df, metric, models, min_count=30):
    """Per functional group, mean of `metric` per model + n (rows, i.e.
    molecule x seed observations, not unique molecules)."""
    exploded = df[["mol_id", "model", "functional_groups", metric]].explode(
        "functional_groups"
    )
    exploded = exploded.dropna(subset=["functional_groups"])
    summary = exploded.pivot_table(
        index="functional_groups",
        columns="model",
        values=metric,
        aggfunc="mean",
    )
    counts = (
        exploded.groupby("functional_groups")["mol_id"].count().rename("n")
    )
    summary = summary.join(counts)
    return summary[summary["n"] >= min_count].round(4)


print(
    "=== Random split, cosine_similarity, by functional group (top 15 by ICICLE) ==="
)
fg_sim_random = functional_group_similarity_summary(
    similarity_long["random"], "cosine_similarity", MODELS
)
display(fg_sim_random.sort_values("ICICLE", ascending=False).head(15))

print()
print(
    "=== Scaffold split, cosine_similarity, by functional group (top 15 by ICICLE) ==="
)
fg_sim_scaffold = functional_group_similarity_summary(
    similarity_long["scaffold"], "cosine_similarity", MODELS
)
display(fg_sim_scaffold.sort_values("ICICLE", ascending=False).head(15))

## Summary table: mean similarity per metric, per model, per split (with n)


In [ ]:
summary_rows = []
for split, df in similarity_long.items():
    for model in MODELS:
        sub = df[df["model"] == model]
        for metric in SIMILARITY_METRICS:
            vals = sub[metric].dropna()
            summary_rows.append(
                {
                    "split": split,
                    "model": model,
                    "metric": metric,
                    "n": len(vals),
                    "mean": vals.mean(),
                    "median": vals.median(),
                }
            )
similarity_summary = pd.DataFrame(summary_rows).round(4)
similarity_summary

---
# Part 5 — Molecular complexity ablation (global retrieval)

Tests a different axis than functional-group rarity above: overall
molecular complexity, via four independent scores (SAScore, NPScore,
SPScore, Boettcher), computed on the same global-retrieval query set.
Complexity features and scoring code adapted from
`/home/magled/one_step_retro_failure_mode/complexity_features`
(now vendored at `src/icicle/utils/chem/complexity/`).

In [ ]:
complexity_df = pd.read_csv(
    OUTPUT_DIR / "merged_top1_structural_global_with_complexity.csv"
)
print(f"{len(complexity_df)} molecules")
complexity_df[["sascore", "npscore", "spscore", "boettcher"]].describe()

## Spearman correlation: complexity score vs. top-1 (per model)

In [ ]:
COMPLEXITY_COLS = {
    "sascore": "SAScore",  # (higher = harder to synthesize)
    "npscore": "NPScore",  # (higher = more natural-product-like)
    "spscore": "SPScore",  # (higher = more spatially/3D complex)
    "boettcher": "Boettcher score",  # (higher = more complex)
}
top1_cols_complexity = {m: f"{m}_top1" for m in GLOBAL_RETRIEVAL_MODELS}

corr_rows = []
for col in COMPLEXITY_COLS:
    sub = complexity_df.dropna(subset=[col])
    row = {"descriptor": col}
    for model, colname in top1_cols_complexity.items():
        rho, p = spearmanr(sub[col], sub[colname].astype(int))
        row[f"{model}_rho"] = rho
        row[f"{model}_p"] = p
    corr_rows.append(row)
complexity_corr_table = pd.DataFrame(corr_rows).round(4)
complexity_corr_table

## Top-1 accuracy by complexity quartile, one panel per descriptor

In [ ]:
quartile_labels = ["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]

for col, col_label in COMPLEXITY_COLS.items():
    sub = complexity_df.dropna(subset=[col]).copy()
    sub["quartile"] = pd.qcut(sub[col], 4, labels=quartile_labels)
    q_summary = sub.groupby("quartile", observed=True)[
        [f"{m}_top1" for m in GLOBAL_RETRIEVAL_MODELS]
    ].mean()

    fig, ax = make_fig("default")
    x = np.arange(len(quartile_labels))
    for model in GLOBAL_RETRIEVAL_MODELS:
        ax.plot(
            x,
            q_summary[f"{model}_top1"] * 100,
            marker="o",
            label=model,
            color=MODEL_COLORS[model],
        )
    ax.set_xticks(x)
    ax.set_xticklabels(quartile_labels, rotation=0, fontsize=7)
    ax.set_xlabel(col_label)
    ax.set_ylabel("Top-1 accuracy (%)")
    ax.legend(frameon=False)
    save_fig(fig, f"global_complexity_{col}_quartile", OUTPUT_DIR)
    print(f"Saved global_complexity_{col}_quartile.[svg|png]")

## Combined 2x2 view (all four complexity descriptors, one figure)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=fig.get_size_inches())
for ax, (col, col_label) in zip(axes.flat, COMPLEXITY_COLS.items()):
    sub = complexity_df.dropna(subset=[col]).copy()
    sub["quartile"] = pd.qcut(sub[col], 4, labels=quartile_labels)
    q_summary = sub.groupby("quartile", observed=True)[
        [f"{m}_top1" for m in GLOBAL_RETRIEVAL_MODELS]
    ].mean()
    x = np.arange(len(quartile_labels))
    for model in GLOBAL_RETRIEVAL_MODELS:
        ax.plot(
            x,
            q_summary[f"{model}_top1"] * 100,
            marker="o",
            markersize=4,
            label=model,
            color=MODEL_COLORS[model],
        )
    ax.set_xticks(x)
    ax.set_xticklabels(quartile_labels, fontsize=6)
    ax.set_title(col_label, fontsize=8)
axes.flat[0].legend(frameon=False, fontsize=6)
for ax in axes[:, 0]:
    ax.set_ylabel("Top-1 accuracy (%)")
fig.tight_layout()
save_fig(fig, "global_complexity_2x2", OUTPUT_DIR)
print("Saved global_complexity_2x2.[svg|png]")

---
# Part 6 — Absolute spectral similarity vs. functional-group rarity

The functional-group top-1 analysis (Part 4) shows NEIMS's *relative*
retrieval advantage grows with a group's query count n. This section
checks the underlying absolute spectral quality (cosine similarity to
ground truth, not rank) for both models as a function of how rare a
molecule's own rarest functional group is, to distinguish two
explanations for the crossover: (1) NEIMS's fingerprint representation
scales especially well with training-data density for a given
chemotype (ICICLE still improves too, just less steeply), vs.
(2) ICICLE has some specific weakness on common/simple structures.
Uses `similarity_results.csv` (mean across 3 seeds) rather than the
top-1/rank data, so this is genuinely a different signal, not a
re-derivation of the retrieval result.

In [ ]:
fg_n_map = global_fg_summary["n"].to_dict()


def min_fg_n(fg_list):
    ns = [fg_n_map[fg] for fg in fg_list if fg in fg_n_map]
    return min(ns) if ns else None


global_merged["min_fg_n"] = global_merged["functional_groups"].apply(min_fg_n)

icicle_sim_abs = (
    pd.concat(
        [
            pd.read_csv(
                d / "similarity_results.csv",
                usecols=["mol_id", "cosine_similarity"],
            )
            for d in SIM_DIRS["random"]["ICICLE"]
        ]
    )
    .groupby("mol_id", as_index=False)["cosine_similarity"]
    .mean()
    .rename(columns={"cosine_similarity": "ICICLE_cosine"})
)
neims_sim_abs = (
    pd.concat(
        [
            pd.read_csv(
                d / "similarity_results.csv",
                usecols=["mol_id", "cosine_similarity"],
            )
            for d in SIM_DIRS["random"]["NEIMS"]
        ]
    )
    .groupby("mol_id", as_index=False)["cosine_similarity"]
    .mean()
    .rename(columns={"cosine_similarity": "NEIMS_cosine"})
)

abs_sim_rarity = (
    global_merged[["mol_id", "min_fg_n"]]
    .merge(icicle_sim_abs, on="mol_id", how="inner")
    .merge(neims_sim_abs, on="mol_id", how="inner")
    .dropna(subset=["min_fg_n"])
)
print(
    f"{len(abs_sim_rarity)} molecules with absolute similarity + FG-rarity data"
)

for col in ["ICICLE_cosine", "NEIMS_cosine"]:
    rho, p = spearmanr(abs_sim_rarity["min_fg_n"], abs_sim_rarity[col])
    print(f"Spearman(min_fg_n, {col}) = {rho:.4f}, p = {p:.2e}")

## Absolute cosine similarity by rarity quartile (of molecule's rarest own FG)

In [ ]:
rarity_labels = ["Q1 (has a rare FG)", "Q2", "Q3", "Q4 (all common FGs)"]
abs_sim_rarity["rarity_q"] = pd.qcut(
    abs_sim_rarity["min_fg_n"], 4, labels=rarity_labels
)
abs_sim_by_rarity = abs_sim_rarity.groupby("rarity_q", observed=True)[
    ["ICICLE_cosine", "NEIMS_cosine"]
].mean()
display(abs_sim_by_rarity.round(4))

fig, ax = make_fig("default")
x = np.arange(len(rarity_labels))
ax.plot(
    x,
    abs_sim_by_rarity["ICICLE_cosine"],
    marker="o",
    label="ICICLE",
    color=MODEL_COLORS["ICICLE"],
)
ax.plot(
    x,
    abs_sim_by_rarity["NEIMS_cosine"],
    marker="o",
    label="NEIMS",
    color=MODEL_COLORS["NEIMS"],
)
ax.set_xticks(x)
ax.set_xticklabels(rarity_labels, fontsize=7)
ax.set_ylabel("Mean cosine similarity to ground truth")
ax.legend(frameon=False)
save_fig(fig, "global_abs_similarity_by_fg_rarity", OUTPUT_DIR)
print("Saved global_abs_similarity_by_fg_rarity.[svg|png]")

## Notes / caveats

- All retrieval comparisons use **top-1 accuracy**, not rank.
- Model colors match `icicle.utils.visualization.eval_plots.model_color`
  (ICICLE=palette[0], NEIMS=palette[3], RASSP=palette[5],
  MassFormer=palette[8]) — same convention as `fig_similarity_results.ipynb`.
- **Every binned table/plot reports n** (query count in that bin) so
  sample size is always visible alongside the summary statistic —
  x-axis tick labels include `(n=...)` on all multi-bin plots, and the
  `n` column appears in every binned DataFrame.
- Global PubChem retrieval (Part 1) is **ICICLE, NEIMS, MassFormer only**
  — RASSP has no PubChem-scale batch-inference prediction HDF5, so no
  global-retrieval comparison is possible for it. Global retrieval scores
  `rank_autofail_cosine == 1` (autofail — a query whose true molecule
  falls outside the candidate window automatically fails at the real
  retrieval task; at the global "all" level, inject and autofail are
  identical since there is no window to be excluded from).
- Formula-match retrieval (Parts 2-3) has no analogous inject/autofail
  distinction — every query's own formula-matched candidate pool
  necessarily contains the true molecule (formula match is symmetric).
- **MassFormer's random-split data (similarity and formula-match) is
  missing seed s2** — 2 seeds instead of 3. `mean_ci_t` handles this
  gracefully (valid t-distribution CI with df=n-1, just wider than the
  3-seed CIs for other models) rather than being excluded.
- Formula-match top-1 uses two aggregations depending on the analysis:
  binned plots compute the rate *per seed* within each bin then apply
  `mean_ci_t` across per-seed bin rates (statistically correct unit for a
  seed CI); functional-group/correlation tables use each query's
  already-seed-averaged `<model>_rate` column (a per-query few-observation
  CI isn't meaningful there — those tables aggregate over many molecules
  per group instead).
- Part 4 (similarity) is a fundamentally different comparison from Parts
  1-3 — it compares predicted-vs-ground-truth spectral similarity
  directly, not retrieval rank/outcome. A molecule can have high predicted
  spectral similarity yet still rank poorly in retrieval if many PubChem
  decoys are even more similar, so Part 4's distributions and Parts 1-3's
  top-1 rates are not expected to track each other exactly.
- `exmol.get_functional_groups(mol, return_all=True)` returns every
  matched group, not just the highest-priority one, so a single molecule
  contributes to multiple functional-group rows.
- ICICLE's checkpoint for the random-split comparisons:
  `entropy_random_s1/checkpoints/best-model-val_loss=0.1183-epoch=48.ckpt`
  (documented in CLAUDE.md as the checkpoint used to build
  `pubchem_predictions_rerun_260710.hdf5`).
- An earlier, INCORRECT similarity comparison used
  `results/eval/icicle_rdm_no_xeno_sim/` (a small, ~3-months-older,
  335-query dev run, not matching the current checkpoint or query-set
  scale) — not used anywhere in this notebook.
